# Phase 7: Unsupervised Anomaly Detection

In this phase, we use the Isolation Forest algorithm to detect abnormal sensor patterns without relying on failure labels. This is critical for detecting unknown or unlabelled faults in industrial systems.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import joblib
from sklearn.ensemble import IsolationForest

# Load dataset
df = pd.read_csv("../datasets/predictive_maintenance.csv")
print("Dataset loaded.")

### Step 84: Select Sensor Features

We select key sensors to monitor for abnormal behavior.

In [ ]:
features = [
    "Air temperature",
    "Process temperature",
    "Rotational speed",
    "Torque",
    "Tool wear"
]

X = df[features]
print("Features selected for unsupervised learning.")

### Step 85 & 86: Train Isolation Forest & Predict Anomalies

In [ ]:
model = IsolationForest(
    contamination=0.02,
    random_state=42
)

model.fit(X)
df["anomaly"] = model.predict(X)

print("Anomaly counts (1 = normal, -1 = anomaly):")
print(df["anomaly"].value_counts())

### Step 88: Visualize Anomalies

In [ ]:
plt.figure(figsize=(12,5))
plt.plot(df["Torque"], label="Torque", alpha=0.5)

anomalies = df[df["anomaly"] == -1]

plt.scatter(
    anomalies.index,
    anomalies["Torque"],
    color="red",
    label="Anomaly",
    s=10
)

plt.legend()
plt.title("Torque Anomaly Detection")
plt.show()

### Step 89: Compare With Real Failures

In [ ]:
print("Cross-tabulation: Real Failures vs Detected Anomalies")
print(pd.crosstab(df["Machine failure"], df["anomaly"]))

### Step 90, 91 & 93: Anomaly Scores & suspicious records

In [ ]:
scores = model.decision_function(X)
df["anomaly_score"] = scores

print("Top 10 Most Suspicious Records:")
print(df.sort_values(by="anomaly_score").head(10))

plt.figure(figsize=(12,5))
plt.hist(df["anomaly_score"], bins=50, color='skyblue', edgecolor='black')
plt.title("Anomaly Score Distribution")
plt.show()

### Step 94: Create Alert Flag

In [ ]:
df["alert"] = df["anomaly_score"] < -0.1
print(f"Number of alerts generated: {df['alert'].sum()}")

### Step 92: Save Anomaly Model

In [ ]:
joblib.dump(model, "../models/isolation_forest.pkl")
print("Anomaly model saved to ../models/isolation_forest.pkl")